# Build US air traffic monthly edge list (i,j,t)

This notebook builds the monthly temporal edge list from the raw US air traffic CSVs.

**Inputs** (from `data/raw/`):
- `US_air_traffic_edges.csv` (large)
- `US_air_traffic_nodes.csv`
- `US_air_traffic_gprops.csv`

**Outputs** (to `data/processed/us_air/`):
- `edges_ijt_monthly.parquet` (preferred) or `edges_ijt_monthly.csv.gz` (fallback)
- `index_monthly.csv`
- `nodes.csv` (+ optional `nodes_mapping.csv`)
- `gprops.csv`

Notes:
- Monthly identifier: `t = year * 100 + month` (e.g., 199002).
- No self-loops. Dedup per (t, i, j).
- Edges remain directed here (no symmetrization).


In [ ]:
# ================================================================
# Setup
# ================================================================
from __future__ import annotations

import time
from pathlib import Path

def find_root_with_src(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'src').is_dir():
            return p
    raise RuntimeError("Directory 'src/' not found.")

ROOT = find_root_with_src(Path.cwd())
import shutil

import numpy as np
import pandas as pd

# ---- paths ------------------------------------------------------
RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed" / "us_air"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

EDGES_PATH = RAW_DIR / "US_air_traffic_edges.csv"
NODES_PATH = RAW_DIR / "US_air_traffic_nodes.csv"
GPROPS_PATH = RAW_DIR / "US_air_traffic_gprops.csv"

# ---- parameters -------------------------------------------------
CHUNKSIZE = 2_000_000  # tune based on RAM

# ---- parquet availability -------------------------------------
try:
    import pyarrow as pa
    import pyarrow.parquet as pq
    PARQUET_OK = True
except Exception:
    PARQUET_OK = False
    pa = None
    pq = None

PARQUET_OK


In [ ]:
# ================================================================
# Load nodes and define mapping (if needed)
# ================================================================
nodes = pd.read_csv(NODES_PATH)
nodes.columns = [c.strip().lstrip("#").strip() for c in nodes.columns]

if "index" in nodes.columns:
    node_ids = nodes["index"].astype(int)
else:
    node_ids = pd.Series(np.arange(len(nodes)), name="index")
    nodes.insert(0, "index", node_ids)

N = int(len(nodes))
unique_ids = np.sort(node_ids.unique())
expected = np.arange(N)

mapping_needed = not np.array_equal(unique_ids, expected)
if mapping_needed:
    id_map = {int(old): int(new) for new, old in enumerate(unique_ids)}
    nodes["mapped_id"] = nodes["index"].map(id_map).astype(int)
    mapping_df = pd.DataFrame({"old_id": unique_ids, "new_id": expected})
    mapping_path = PROCESSED_DIR / "nodes_mapping.csv"
    mapping_df.to_csv(mapping_path, index=False)
    print(f"Saved mapping to {mapping_path}")
else:
    id_map = None

nodes_out = PROCESSED_DIR / "nodes.csv"
nodes.to_csv(nodes_out, index=False)
print(f"Saved nodes to {nodes_out} (N={N})")


In [ ]:
# ================================================================
# Copy gprops metadata
# ================================================================
if GPROPS_PATH.exists():
    gprops_out = PROCESSED_DIR / "gprops.csv"
    shutil.copy(GPROPS_PATH, gprops_out)
    print(f"Copied gprops to {gprops_out}")
else:
    print("Warning: gprops file not found, skipping copy.")


In [ ]:
# ================================================================
# Process edges in chunks and write staging partitions by month
# ================================================================
def _clean_columns(cols):
    return [c.strip().lstrip("#").strip() for c in cols]

usecols = lambda c: c.strip().lstrip("#").strip() in {"source", "target", "year", "quarter", "month"}

staging_dir = PROCESSED_DIR / "_staging_edges_by_t"
if staging_dir.exists():
    staging_dir = PROCESSED_DIR / f"_staging_edges_by_t_{int(time.time())}"
staging_dir.mkdir(parents=True, exist_ok=True)
print(f"Staging dir: {staging_dir}")

quarter_map: dict[int, int] = {}
rows_seen = 0

reader = pd.read_csv(
    EDGES_PATH,
    usecols=usecols,
    chunksize=CHUNKSIZE,
    dtype="int32",
)

for chunk_idx, chunk in enumerate(reader, start=1):
    chunk.columns = _clean_columns(chunk.columns)
    chunk = chunk.rename(columns={"source": "i", "target": "j"})

    if id_map is not None:
        chunk["i"] = chunk["i"].map(id_map).astype("int32")
        chunk["j"] = chunk["j"].map(id_map).astype("int32")

    # remove self-loops
    chunk = chunk[chunk["i"] != chunk["j"]]

    # monthly id
    chunk["t"] = (chunk["year"] * 100 + chunk["month"]).astype("int32")

    # update quarter map
    for t_val, q_val in chunk[["t", "quarter"]].drop_duplicates().itertuples(index=False):
        q_val = int(q_val)
        if t_val in quarter_map and quarter_map[t_val] != q_val:
            raise ValueError(f"Quarter mismatch for t={t_val}: {quarter_map[t_val]} vs {q_val}")
        quarter_map[t_val] = q_val

    chunk = chunk[["i", "j", "t"]]
    chunk = chunk.drop_duplicates(["t", "i", "j"])

    for t_val, sub in chunk.groupby("t", sort=False):
        out_path = staging_dir / f"t={int(t_val)}.csv"
        header = not out_path.exists()
        sub.to_csv(out_path, mode="a", header=header, index=False)

    rows_seen += len(chunk)
    if chunk_idx % 5 == 0:
        print(f"Chunks processed: {chunk_idx} | rows kept so far: {rows_seen}")

print(f"Total rows kept after chunk dedup: {rows_seen}")


In [ ]:
# ================================================================
# Dedup per month and write final edge list
# ================================================================
t_files = sorted(staging_dir.glob("t=*.csv"), key=lambda p: int(p.stem.split("=")[1]))
if not t_files:
    raise RuntimeError("No staging files found; check inputs.")

edge_counts: dict[int, int] = {}

if PARQUET_OK:
    edges_out = PROCESSED_DIR / "edges_ijt_monthly.parquet"
    schema = pa.schema([
        ("i", pa.int32()),
        ("j", pa.int32()),
        ("t", pa.int32()),
    ])
    writer = None

    for path in t_files:
        t_val = int(path.stem.split("=")[1])
        df = pd.read_csv(path, dtype={"i": "int32", "j": "int32", "t": "int32"})
        df = df.drop_duplicates(["t", "i", "j"])
        if df.duplicated(['t', 'i', 'j']).any():
            raise ValueError(f"Duplicates remain after dedup for t={t_val}")
        if (df["i"] == df["j"]).any():
            raise ValueError(f"Self-loop found in t={t_val}")
        edge_counts[t_val] = int(len(df))
        table = pa.Table.from_pandas(df, schema=schema, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(edges_out, schema=schema)
        writer.write_table(table)

    if writer is not None:
        writer.close()
else:
    import gzip

    edges_out = PROCESSED_DIR / "edges_ijt_monthly.csv.gz"
    with gzip.open(edges_out, "wt", newline="") as gz:
        gz.write("i,j,t\n")
        for path in t_files:
            t_val = int(path.stem.split("=")[1])
            df = pd.read_csv(path, dtype={"i": "int32", "j": "int32", "t": "int32"})
            df = df.drop_duplicates(["t", "i", "j"])
        if df.duplicated(['t', 'i', 'j']).any():
            raise ValueError(f"Duplicates remain after dedup for t={t_val}")
            if (df["i"] == df["j"]).any():
                raise ValueError(f"Self-loop found in t={t_val}")
            edge_counts[t_val] = int(len(df))
            df.to_csv(gz, header=False, index=False)

print(f"Saved edges to {edges_out}")


In [ ]:
# ================================================================
# Build index_monthly.csv
# ================================================================
rows = []
for t_val in sorted(edge_counts):
    year = int(t_val // 100)
    month = int(t_val % 100)
    quarter = int(quarter_map.get(t_val, (month - 1) // 3 + 1))
    start_date = pd.Timestamp(year=year, month=month, day=1)
    end_date = (start_date + pd.offsets.MonthEnd(1)).date().isoformat()
    rows.append({
        "t": int(t_val),
        "year": year,
        "month": month,
        "quarter": quarter,
        "start_date": start_date.date().isoformat(),
        "end_date": end_date,
        "n_nodes": int(N),
        "n_edges_directed": int(edge_counts[t_val]),
    })

index_df = pd.DataFrame(rows).sort_values("t").reset_index(drop=True)
index_out = PROCESSED_DIR / "index_monthly.csv"
index_df.to_csv(index_out, index=False)
print(f"Saved index to {index_out}")


In [ ]:
# ================================================================
# Sanity checks
# ================================================================
print(f"N nodes: {N}")
print(f"Months (T): {len(index_df)}")
print(f"t range: {index_df['t'].min()} -> {index_df['t'].max()}")
print(f"date range: {index_df['start_date'].min()} -> {index_df['end_date'].max()}")

edges_stats = index_df['n_edges_directed']
print("n_edges_directed per month (min/median/max):",
      int(edges_stats.min()), int(edges_stats.median()), int(edges_stats.max()))

print("Self-loops removed and duplicates checked per month during write.")
